In [29]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import squareform

from kmodes.kprototypes import KPrototypes
from kneed import KneeLocator
from sklearn.metrics import adjusted_rand_score, silhouette_score

In [ ]:
# Load your dataset 
# Replace the file name as required
df = pd.read_excel("your_dataset.xlsx")

# Remove leading/trailing whitespace from column names
df.columns = df.columns.str.strip()

In [ ]:
# Define clustering variables 

id_col = "ID"
num_cols = ["WWR", "SVR", "AR"]  
cat_cols = ["Height","Wtype", "Stype"]


In [ ]:
# data preparation 
df_clean = (
    df.dropna(subset=num_cols + cat_cols)
    .reset_index(drop=True)
    .copy()
)

df_clean[num_cols] = df_clean[num_cols].astype(float)

for col in cat_cols:
    df_clean[col] = (
        df_clean[col]
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r"[\\]+", "", regex=True)
        .str.replace(r"\s+", " ", regex=True)
    )

print("Number of observations after cleaning:", len(df_clean))


In [ ]:
# common dataset for clustering comparison 
X_compare = df_clean[num_cols + cat_cols].copy()

print("Number of observations:", len(X_compare))
print("Variables:", X_compare.columns.tolist())

In [ ]:
# 1. k-prototype clustering 

In [ ]:
X = X_compare.to_numpy()
categorical_idx = [X_compare.columns.get_loc(col) for col in cat_cols]

print("Categorical column indices:", categorical_idx)

In [ ]:
# Run final k-prototypes model
kp_final = KPrototypes(
    n_clusters=4,
    init="Huang",
    n_init=30,
    gamma=0.5,
    random_state=42,
    verbose=0
)

labels_kproto = kp_final.fit_predict(
    X,
    categorical=categorical_idx
)
df_kproto = df_clean.copy()
df_kproto["Cluster"] = labels_kproto

profile_kproto = (
    df_kproto
    .groupby("Cluster")
    .agg({
        "WWR": "mean",
        "SVR": "mean",
        "AR": "mean",
        "Height": lambda x: x.mode().iloc[0],
        "Wtype": lambda x: x.mode().iloc[0],
        "Stype": lambda x: x.mode().iloc[0],
        "ID": "count"
    })
    .rename(columns={"ID": "n"})
    .round(3)
)

profile_kproto

In [ ]:
# Gower distance 

In [ ]:
def gower_distance_matrix(data, num_cols, cat_cols):

    n = len(data)
    D = np.zeros((n, n))

    # Numerical variables
    for col in num_cols:
        values = data[col].astype(float).to_numpy()
        col_range = values.max() - values.min()

        if col_range == 0:
            continue

        D += np.abs(values[:, None] - values[None, :]) / col_range

    # Categorical variables
    for col in cat_cols:
        values = data[col].astype(str).to_numpy()
        D += (values[:, None] != values[None, :]).astype(float)

    # Average across all variables
    D /= (len(num_cols) + len(cat_cols))

    return D


D_gower = gower_distance_matrix(X_compare, num_cols, cat_cols)

print("Shape:", D_gower.shape)

In [ ]:
# 2. K-mediod 

In [ ]:
# define the algorithm 

def pam_kmedoids(D, k, random_state=42, max_iter=100):

    rng = np.random.default_rng(random_state)
    n = D.shape[0]

    medoids = np.sort(rng.choice(n, size=k, replace=False))

    def assign_and_cost(medoids):
        distances = D[:, medoids]
        labels = np.argmin(distances, axis=1)
        cost = np.sum(distances[np.arange(n), labels])
        return labels, cost

    labels, current_cost = assign_and_cost(medoids)

    for _ in range(max_iter):

        best_cost = current_cost
        best_medoids = medoids.copy()
        non_medoids = np.setdiff1d(np.arange(n), medoids)

        for medoid_position in range(k):
            for candidate in non_medoids:

                trial = medoids.copy()
                trial[medoid_position] = candidate
                trial = np.sort(trial)

                _, trial_cost = assign_and_cost(trial)

                if trial_cost < best_cost:
                    best_cost = trial_cost
                    best_medoids = trial.copy()

        if np.array_equal(best_medoids, medoids):
            break

        medoids = best_medoids
        current_cost = best_cost

    labels, final_cost = assign_and_cost(medoids)

    return labels, medoids, final_cost

In [ ]:
# determine k 

k_values = range(2, 11)   
seeds = [0, 10, 20, 42, 99]

best_costs = []

for k in k_values:

    costs_k = []

    for seed in seeds:
        labels, medoids, cost = pam_kmedoids(
            D_gower,
            k=k,
            random_state=seed,
            max_iter=100
        )

        costs_k.append(cost)

    best_costs.append(min(costs_k))


elbow_results = pd.DataFrame({
    "k": list(k_values),
    "Best_cost": best_costs})

kneedle = KneeLocator(
    list(k_values),
    best_costs,
    curve="convex",
    direction="decreasing"
)
print("Kneedle selected k =", kneedle.knee)


In [ ]:
# k-mediod model at selected k=5 

k_final = 5
seeds = [0, 10, 20, 42, 99]

kmedoid_runs = []

for seed in seeds:
    labels, medoids, cost = pam_kmedoids(
        D_gower,
        k=k_final,
        random_state=seed,
        max_iter=100
    )

    kmedoid_runs.append({
        "seed": seed,
        "labels": labels,
        "medoids": medoids,
        "cost": cost
    })

# Select the lowest-cost solution
best_run = min(kmedoid_runs, key=lambda x: x["cost"])

labels_kmedoids = best_run["labels"]
medoid_indices = best_run["medoids"]
final_cost = best_run["cost"]
best_seed = best_run["seed"]
df_kmedoids = df_clean.copy()
df_kmedoids["Cluster"] = labels_kmedoids

profile_kmedoids = (
    df_kmedoids
    .groupby("Cluster")
    .agg({
        "WWR": "mean",
        "SVR": "mean",
        "AR": "mean",
        "Height": lambda x: x.mode().iloc[0],
        "Wtype": lambda x: x.mode().iloc[0],
        "Stype": lambda x: x.mode().iloc[0],
        "ID": "count"
    })
    .rename(columns={"ID": "n"})
    .round(3)
)

profile_kmedoids


In [ ]:
def cluster_balance(labels):
    sizes = pd.Series(labels).value_counts().sort_index()

    return {
        "balance_ratio": sizes.min() / sizes.max()
    }

balance_kmedoids = cluster_balance(labels_kmedoids)
balance_kmedoids

In [ ]:
silhouette_kmedoids = silhouette_score(
    D_gower,
    labels_kmedoids,
    metric="precomputed"
)

print("K-medoids Gower silhouette:", round(silhouette_kmedoids, 3))

In [ ]:
# k-medoids stability- 80% of the sample 

rng = np.random.default_rng(2026)

n = len(df_clean)
n_sub = int(round(0.80 * n))

n_repeats = 100
pam_seeds = [0, 10, 20, 42, 99]

ari_kmedoids_stability = []

for rep in range(n_repeats):

    idx = np.sort(
        rng.choice(n, size=n_sub, replace=False)
    )

    sub_df = df_clean.iloc[idx].reset_index(drop=True)

    D_sub = gower_distance_matrix(
        sub_df,
        num_cols,
        cat_cols
    )

    best_cost = np.inf
    best_labels = None

    for seed in pam_seeds:

        labels_temp, medoids_temp, cost_temp = pam_kmedoids(
            D_sub,
            k=5,
            random_state=seed,
            max_iter=100
        )

        if cost_temp < best_cost:
            best_cost = cost_temp
            best_labels = labels_temp

    ari = adjusted_rand_score(
        labels_kmedoids[idx],
        best_labels
    )

    ari_kmedoids_stability.append(ari)

print("K-medoids stability at 80% retention")
print("Mean ARI:", round(np.mean(ari_kmedoids_stability), 3))


In [ ]:
# 3. Hierarchical agglomerative clustering 

In [ ]:
# Convert Gower matrix to condensed form
D_condensed = squareform(D_gower, checks=False)

# Build HAC hierarchy using average linkage
Z = linkage(D_condensed, method="average")

In [ ]:
plt.figure(figsize=(16, 8))

dendrogram(
    Z,
    labels=df_clean["ID"].astype(str).to_numpy(),
    leaf_rotation=90,
    leaf_font_size=8
)

plt.xlabel("Building ID")
plt.ylabel("Gower dissimilarity")
plt.title("HAC dendrogram: Gower distance + average linkage")

plt.tight_layout()
plt.show()


In [ ]:
linkage_distances = Z[:, 2]
jumps = np.diff(linkage_distances)

n = len(df_clean)

jump_results = pd.DataFrame({
    "Distance_before": linkage_distances[:-1],
    "Distance_after": linkage_distances[1:],
    "Jump": jumps,
    "k": np.arange(n - 1, 1, -1)
})

jump_results = jump_results.sort_values(
    "Jump",
    ascending=False
).reset_index(drop=True)

jump_results.head()

In [ ]:
# candidate cluster profiles 

def cluster_profiles(data, labels, num_cols, cat_cols):
    temp = data.copy()
    temp["Cluster"] = labels

    numeric = (
        temp.groupby("Cluster")[num_cols]
        .mean()
        .round(3)
    )

    categorical = (
        temp.groupby("Cluster")[cat_cols]
        .agg(lambda x: x.mode().iloc[0])
    )

    n = (
        temp.groupby("Cluster")
        .size()
        .rename("n")
    )

    return pd.concat([numeric, categorical, n], axis=1)


# Examine candidate solutions identified from the dendrogram
hac_solutions = {}

for k in [2, 3, 4]:
    labels = fcluster(
        Z,
        t=k,
        criterion="maxclust"
    )

    hac_solutions[k] = labels

    print(f"\nHAC: {k}-CLUSTER SOLUTION")
    display(
        cluster_profiles(
            df_clean,
            labels,
            num_cols,
            cat_cols
        )
    )

In [ ]:
# final HAC model with k=4

labels_hac = hac_solutions[4]
profile_hac = cluster_profiles(
    df_clean,
    labels_hac,
    num_cols,
    cat_cols
)

profile_hac


In [ ]:
cluster_sizes_hac = pd.Series(labels_hac).value_counts().sort_index()

balance_hac = (
    cluster_sizes_hac.min()
    / cluster_sizes_hac.max()
)
print("\nHAC balance ratio:", round(balance_hac, 3))

In [ ]:
silhouette_hac = silhouette_score(
    D_gower,
    labels_hac,
    metric="precomputed"
)

print("HAC Gower silhouette:", round(silhouette_hac, 3))

In [ ]:
# HAC stability- 80% of the sample 

rng = np.random.default_rng(2026)

n = len(df_clean)
n_sub = int(round(0.80 * n))
n_repeats = 100

ari_hac_stability = []

for rep in range(n_repeats):

    idx = np.sort(rng.choice(n, size=n_sub, replace=False))
    sub_df = df_clean.iloc[idx].reset_index(drop=True)

    D_sub = gower_distance_matrix(
        sub_df,
        num_cols,
        cat_cols
    )

    D_sub_condensed = squareform(D_sub, checks=False)

    Z_sub = linkage(
        D_sub_condensed,
        method="average"
    )

    labels_sub = fcluster(
        Z_sub,
        t=4,
        criterion="maxclust"
    )

    ari = adjusted_rand_score(
        labels_hac[idx],
        labels_sub
    )

    ari_hac_stability.append(ari)

print("HAC stability at 80% retention")
print("Mean ARI:", round(np.mean(ari_hac_stability), 3))
